# Migration Phase 0: Setup Check

Verifies the three new data sources are reachable before we build loaders against them:

1. `brainlink` package + DB (behavioral/demographic data)
2. Tabular neuroimaging derivatives root (anatomical + diffusion)
3. `regional-stacker` package (multivariate regional BAG modeling)

See `data_pipeline_migration_plan.md` for the full plan and resolved decisions.

## 1. Imports

Both new dependencies were added to `pyproject.toml` as local path deps:

```toml
"brainlink @ file:///home/galkepler/Projects/brainlink",
"regional-stacker @ file:///home/galkepler/Projects/regional-stacker",
```

(Required adding `[tool.hatch.metadata] allow-direct-references = true`, since hatchling rejects direct path/URL references by default.)

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from brainlink import BrainLinkDB
from regional_stacker import RegionalStackingRegressor, wide_to_stacker_input

print("brainlink + regional-stacker import OK")

brainlink + regional-stacker import OK


## 2. Behavioral data (brainlink)

`BRAINLINK_DB_PATH` in `.env` points at the brainlink SQLite DB. `db.info()` prints a summary of
what's in it (participants, sessions, demographics, questionnaires, imaging paths).

In [2]:
db_path = os.environ["BRAINLINK_DB_PATH"]
assert Path(db_path).exists(), f"brainlink DB not found at {db_path}"

db = BrainLinkDB(db_path)
db.info()

BrainLinkDB
  Path : /media/storage/brainlink/brainlink.db
  Size : 91.1 MB

  Table counts
    sessions              : 9,953
    participants          : 5,526
    questionnaire resp.   : 6,058
    imaging paths         : 0
    questionnaire columns : 186

  Completeness
    mapping_complete      : 4,893 / 9,953  (49.2%)
    has_demographics      : 5,005 / 9,953  (50.3%)
    has_questionnaire     : 4,167 subjects

  Lab breakdown
                          4955  █████████
    YA                    3065  ██████
    SNBB                  1320  ██
    YBH                    267  
    IT                     241  
    TS                      44  
    YG                      17  
    MG                      17  
    TBS                      9  
    GY                       7  
    YY                       6  
    LM                       5  

  Last ingest : 2026-06-02 10:04:05.783281  (job=refresh_sheets, run=528248fd-25c6-4d28-9b53-4d6680311f2d)


### Note: `BRAINLINK_DB_PATH`

The neuroalign-local copy at `/home/galkepler/Projects/brainlink/brainlink.db` is empty (0
rows in every table) and has a stale schema (`participant_id` instead of `uid`) relative to
the installed `brainlink` package's ORM.

The real, populated DB lives at `/media/storage/brainlink/brainlink.db` (9,953 sessions,
5,526 participants, schema matches `models.py`). `.env`'s `BRAINLINK_DB_PATH` points there.

## 3. Tabular neuroimaging derivatives

`TABULAR_DERIVATIVES_ROOT` points at `/mnt/62/Processed_Data/derivatives/tabular`, one
`sub-<UID>/` directory per participant. Each subject has a subject-level `anat/`, plus
per-session `ses-<id>/`, `ses-<id>.cross/` directories with `anat/` and `dwi/` subfolders.

Default config (see `data_pipeline_migration_plan.md`):
- `ATLAS_NAME=Schaefer2018N400n7Tian2020S2` (dwi naming)
- `ANAT_ATLASES=Schaefer2018N400n7,Tian2020S2` (anat cortex+subcortex, concatenated)
- `SESSION_VARIANT=cross` (prefer `ses-<id>.cross/` for max session coverage)

In [3]:
tabular_root = Path(os.environ["TABULAR_DERIVATIVES_ROOT"])
assert tabular_root.exists(), f"tabular derivatives root not found at {tabular_root}"

subject_dirs = sorted(tabular_root.glob("sub-*"))
print(f"{len(subject_dirs)} subject directories found")
print("example:", subject_dirs[0].name)

# spot-check: does the example subject have the expected anat atlases?
example = subject_dirs[0]
for atlas in os.environ["ANAT_ATLASES"].split(","):
    p = example / "anat" / f"atlas-{atlas}"
    print(atlas, "->", "OK" if p.exists() else "MISSING", p)

2828 subject directories found
example: sub-S000091


Schaefer2018N400n7 -> OK /mnt/62/Processed_Data/derivatives/tabular/sub-S000091/anat/atlas-Schaefer2018N400n7
Tian2020S2 -> OK /mnt/62/Processed_Data/derivatives/tabular/sub-S000091/anat/atlas-Tian2020S2


## 4. Cross-check: brainlink sessions vs. tabular derivatives

For each session with a complete `uid`/`subject_code`/`session_id` mapping, check whether
the corresponding `sub-<uid>/ses-<session_id>{.cross,}/` directory exists under
`TABULAR_DERIVATIVES_ROOT`.

In [4]:
sessions = db.query(include=["demographics"], require_complete_mapping=True, styled=False)
print(f"{len(sessions)} sessions with complete uid/subject_code/session_id mapping")

sample = sessions.sample(10, random_state=42)
for _, row in sample.iterrows():
    sub_dir = tabular_root / f"sub-{row['uid']}"
    cross_dir = sub_dir / f"ses-{row['session_id']}.cross"
    plain_dir = sub_dir / f"ses-{row['session_id']}"
    print(
        row["uid"], row["session_id"],
        "cross:", cross_dir.exists(),
        "plain:", plain_dir.exists(),
    )

4893 sessions with complete uid/subject_code/session_id mapping
S887763 202409111849 cross: True plain: True
S596906 202212061129 cross: False plain: True
S045146 202202141732 cross: True plain: True
S809920 202409302005 cross: True plain: True
S997394 202602231308 cross: False plain: False
S330623 202209182005 cross: False plain: True
S791080 202311141355 cross: False plain: True
S168352 202602011301 cross: False plain: False
S389944 202410091603 cross: False plain: True
S394518 202106091559 cross: False plain: True


## 5. Note on unrelated dependency drift

`uv sync --all-extras` removed several packages that were present in the venv but never
declared in `pyproject.toml` (e.g. `umap-learn`, `streamlit`, `pyvista`, `surfplot`, `vtk`,
`subcortex-visualization`). These are used by `src/neuroalign/visualization/brain.py` and
`app/main.py` (separate WIP, unrelated to this migration). This drift pre-dates this branch
(confirmed against the prior `uv.lock`) — flagged here, not fixed as part of this migration.